[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ray-certified/notebooks/day-08-ray-workflows-jobs.ipynb#scrollTo=10a2b3c4)

---
# Day 8 · Ray Workflows and Job Submission
**certified-journeys / ray-certified** · Review Day

> **Goal for today:** Convert a multi-step pipeline into a durable Ray Workflow with checkpointing, and understand the Ray Jobs API for submitting and managing workloads on a Ray cluster.


In [ ]:
%pip install -q 'ray[default]'


## Concept 1 · The Ray Jobs API

The **Ray Jobs API** is the production mechanism for running Python scripts on a Ray cluster without attaching interactively. It solves three problems:

| Problem | Ray Jobs solution |
|---|---|
| Running scripts on a remote cluster | `ray job submit --address <head> -- python my_script.py` |
| Watching logs without SSH | `ray job logs <job_id>` (streaming) |
| Cancelling runaway jobs | `ray job stop <job_id>` |
| Inspecting job status | `ray job status <job_id>` |

### Job lifecycle

```
PENDING → RUNNING → SUCCEEDED
                  ↘ FAILED
                  ↘ STOPPED  (via ray job stop)
```

In a **notebook / local cluster**, the Python SDK (`JobSubmissionClient`) replaces the CLI:
```python
from ray.job_submission import JobSubmissionClient
client = JobSubmissionClient("http://127.0.0.1:8265")  # dashboard address
job_id = client.submit_job(entrypoint="python my_script.py")
```


## Concept 2 · Submitting and Monitoring a Job Programmatically

The `JobSubmissionClient` lets you:
- **Submit** a job with an entrypoint command and optional runtime environment
- **Poll** status until `SUCCEEDED` or `FAILED`
- **Stream logs** in real time
- **Stop** a running job

A **runtime environment** bundles the job's dependencies so remote workers can resolve them:
```python
runtime_env = {
    "pip": ["scikit-learn", "pandas"],
    "env_vars": {"MY_PARAM": "42"},
    "working_dir": "./my_project",  # uploaded to the cluster
}
```


In [ ]:
import ray
import time
import tempfile
import os

# Start a local Ray cluster and wait for the dashboard to come up
ray.init(ignore_reinit_error=True, include_dashboard=True)
time.sleep(2)  # give the dashboard HTTP server a moment

from ray.job_submission import JobSubmissionClient, JobStatus

# Connect to the local cluster's job submission endpoint
client = JobSubmissionClient("http://127.0.0.1:8265")

# ── Write a tiny Python script to submit as a job ────────────────────────────
job_script = """
import ray, time
ray.init(address='auto')  # connect to the existing cluster

@ray.remote
def square(x):
    return x * x

results = ray.get([square.remote(i) for i in range(5)])
print('Job result:', results)
"""

# Write script to a temp directory (the working_dir that gets uploaded)
tmpdir = tempfile.mkdtemp()
script_path = os.path.join(tmpdir, "ray_job.py")
with open(script_path, "w") as f:
    f.write(job_script)

# Submit the job — entrypoint is a shell command
job_id = client.submit_job(
    entrypoint="python ray_job.py",
    runtime_env={"working_dir": tmpdir},
)
print(f"Submitted job: {job_id}")


### What just happened?
- **`JobSubmissionClient`** connected to the Ray head node's REST API (port 8265 is the dashboard / job submission port).
- **`submit_job`** uploaded the `working_dir`, registered the job, and returned a `job_id` immediately — the job runs asynchronously.
- **`runtime_env.working_dir`** is zipped, checksummed, and distributed to every worker node that runs tasks from this job.
- In production, `address` points to your cluster head (e.g., `ray://cluster-head:10001` or the Kubernetes service address).


In [ ]:
# Poll until the job finishes and print logs
import time

def wait_for_job(client, job_id, timeout=60):
    """Poll job status every 2 s; print final logs when done."""
    start = time.time()
    while time.time() - start < timeout:
        status = client.get_job_status(job_id)
        print(f"  [{time.strftime('%H:%M:%S')}] Status: {status}")
        if status in (JobStatus.SUCCEEDED, JobStatus.FAILED, JobStatus.STOPPED):
            break
        time.sleep(2)
    logs = client.get_job_logs(job_id)
    print("\n── Job logs ─────────────────────")
    print(logs)
    return status

final_status = wait_for_job(client, job_id)
print(f"\nFinal status: {final_status}")


### What just happened?
- **`client.get_job_status(job_id)`** is the SDK equivalent of `ray job status <id>` — returns a `JobStatus` enum.
- **`client.get_job_logs(job_id)`** fetches all stdout/stderr produced by the job's entrypoint — equivalent to `ray job logs <id>`.
- To stream logs in real time (instead of fetching all at once), use `client.tail_job_logs(job_id)` which returns an iterator.
- To cancel: `client.stop_job(job_id)` — equivalent to `ray job stop <id>` on the CLI.


## Concept 3 · Ray Workflows — Durable Multi-Step Pipelines

Ray Tasks are ephemeral: if the driver dies mid-run, all results are lost. **Ray Workflows** add durability:

| Property | Ray Tasks | Ray Workflows |
|---|---|---|
| Checkpointing | None | Automatic after every step |
| Re-run after failure | From scratch | Resume from last checkpoint |
| Unique ID per run | None | User-supplied `workflow_id` |
| Step retry on failure | Manual | Configurable via `max_retries` |

### When to use Workflows vs raw Tasks

- **Workflows**: multi-hour pipelines, fan-out then fan-in, must be restartable
- **Raw Tasks**: sub-minute functions, interactive Jupyter, don't need checkpointing

The API looks nearly identical to `@ray.remote`, with `@workflow.step` instead:


## Concept 4 · `@workflow.step` Decorator and `workflow.run()`

```python
from ray import workflow

@workflow.step
def my_step(x):
    return x * 2

# Calling step.step() returns a WorkflowOutput, not the actual result
output = my_step.step(5)

# workflow.run() executes the graph synchronously, with checkpointing
result = workflow.run(output, workflow_id="my-unique-run")
```

Key difference vs `ray.remote`:
- `.step()` instead of `.remote()`
- `workflow.run()` instead of `ray.get()`
- Pass `workflow_id` to resume on failure: `workflow.resume("my-unique-run")`


In [ ]:
from ray import workflow

# ── Three-step ML-style pipeline ─────────────────────────────────────────────

@workflow.step
def ingest(n_samples: int) -> dict:
    """Step 1: simulate data loading."""
    import random, time
    time.sleep(0.5)  # simulate I/O
    data = [random.gauss(0, 1) for _ in range(n_samples)]
    print(f"[ingest] Loaded {len(data)} samples")
    return {"data": data, "n": n_samples}


@workflow.step
def transform(payload: dict) -> dict:
    """Step 2: compute mean and std."""
    import statistics
    data = payload["data"]
    result = {
        "mean": statistics.mean(data),
        "stdev": statistics.stdev(data),
        "n": payload["n"],
    }
    print(f"[transform] mean={result['mean']:.4f}, stdev={result['stdev']:.4f}")
    return result


@workflow.step
def report(stats: dict) -> str:
    """Step 3: format a human-readable summary."""
    summary = (
        f"Pipeline report\n"
        f"  Samples : {stats['n']}\n"
        f"  Mean    : {stats['mean']:.4f}\n"
        f"  Std Dev : {stats['stdev']:.4f}\n"
    )
    print(summary)
    return summary


# Build the workflow graph (nothing executes yet)
raw_data  = ingest.step(1000)
stats     = transform.step(raw_data)
final     = report.step(stats)

# Run the workflow — checkpoints after each step
result = workflow.run(final, workflow_id="stats-pipeline-v1")
print("Workflow result:\n", result)


### What just happened?
- **`.step()`** creates a `WorkflowOutput` node — a lazy description of work, exactly analogous to `.remote()`.
- **`workflow.run()`** traverses the graph topologically, executes each step in order, and checkpoints the output to Ray's object store + a persistent workflow store before moving to the next step.
- If the Python process dies after step 1 completes, calling `workflow.resume("stats-pipeline-v1")` would skip step 1 and resume from step 2 — that checkpoint is already saved.
- The `workflow_id` is your durability anchor: identical IDs let Workflows idempotently resume.


## Concept 5 · Workflow Durability — Checkpointing and Resume

By default, Ray Workflows store checkpoints in `/tmp/ray/workflow_data/`. In production, configure a remote backend:

```python
ray.init(
    storage="s3://my-bucket/workflow-data",  # S3, GCS, or Azure Blob
)
```

### Resuming after failure

```python
# If the run crashed mid-way:
result = workflow.resume("stats-pipeline-v1")

# List all known workflows:
all_wf = workflow.list_all()
print(all_wf)  # {"stats-pipeline-v1": WorkflowStatus.SUCCESSFUL}
```

### Step-level retry

```python
@workflow.step(max_retries=3, catch_exceptions=True)
def flaky_step(x):
    ...
```

With `catch_exceptions=True`, the step returns `(result, None)` on success or `(None, exception)` on failure, letting the next step decide how to handle it.


In [ ]:
# Inspect workflow status programmatically
all_workflows = workflow.list_all()
print("Known workflows:")
for wf_id, wf_status in all_workflows.items():
    print(f"  {wf_id}: {wf_status}")

# Retrieve the result of a completed workflow without re-running it
cached_result = workflow.get_output("stats-pipeline-v1")
print("\nCached workflow output:\n", cached_result)


### What just happened?
- **`workflow.list_all()`** queries the workflow store and returns all workflow IDs with their final status.
- **`workflow.get_output(workflow_id)`** fetches the stored result without re-executing any steps — the checkpoint is the source of truth.
- In production this is valuable for **idempotent pipelines**: check `list_all()` first; if the workflow already succeeded, fetch the output instead of re-running.


## Concept 6 · Ray Workflows vs Raw Tasks — Decision Guide

| Criterion | Use Raw Tasks | Use Ray Workflows |
|---|---|---|
| Pipeline duration | < 5 minutes | Hours / days |
| Need to resume on failure | No | Yes |
| Audit trail of intermediate results | No | Yes |
| Overhead acceptable | N/A | ~100 ms per step |
| Dynamic branching (if/else in the DAG) | Easy (Python) | Supported via `workflow.continuation` |

The checkpoint overhead (~100 ms per step) is negligible for hour-long pipelines but significant for sub-second tasks. Prefer raw `@ray.remote` tasks for tight loops and use Workflows as an orchestration layer around coarser-grained steps.


## Concept 7 · Dynamic Workflows with `workflow.continuation`

A workflow step can return another step — this is how you build dynamic DAGs (branches, recursion):

```python
@workflow.step
def decide(data):
    if needs_cleaning(data):
        return workflow.continuation(clean.step(data))
    return data
```

The returned `WorkflowOutput` is transparently injected into the workflow graph at runtime — no static DAG declaration needed. This enables:
- **Conditional branches** (if quality check fails, run cleaning)
- **Recursive workflows** (retry a step with different params)
- **Fan-out** (spawn N parallel steps, then aggregate)


In [ ]:
# Challenge: Build a workflow with a conditional branch
#
# Step 1: generate_numbers(n) → returns a list of n random integers (0–100)
# Step 2: quality_check(numbers) →
#   - if mean > 50: return {"status": "good", "mean": mean}
#   - if mean <= 50: use workflow.continuation to call resample.step(n)
# Step 3: resample(n) → generate_numbers again (new random seed)
#   (resample just calls generate_numbers.step(n) to loop)
# Step 4: summarise(result) → print and return a string summary
#
# Run with workflow.run(summarise.step(quality_check.step(generate_numbers.step(20))),
#                       workflow_id="conditional-pipeline-v1")

# Your solution here:
# @workflow.step
# def generate_numbers(n): ...

# @workflow.step
# def quality_check(numbers): ...

# @workflow.step
# def resample(n): ...

# @workflow.step
# def summarise(result): ...


In [ ]:
# Cleanup
ray.shutdown()
print("Ray shut down.")


---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| Ray Jobs API | CLI or SDK to submit scripts to a cluster; track status, logs, cancel |
| `JobSubmissionClient` | Python SDK for the jobs REST API; connects to dashboard port 8265 |
| `runtime_env` | Bundles pip deps + env vars + working_dir; distributed to workers |
| `@workflow.step` | Like `@ray.remote` but adds checkpointing; use `.step()` not `.remote()` |
| `workflow.run()` | Executes the workflow DAG synchronously; saves checkpoint after each step |
| `workflow.resume()` | Re-runs from last checkpoint — skip already-completed steps |
| `workflow.continuation` | Return a step from a step to build dynamic/conditional DAGs |
| Workflows vs Tasks | Workflows for multi-hour restartable pipelines; Tasks for sub-minute work |

> **Tip:** Ray Workflows checkpoint after every step — adds latency but makes the pipeline restartable. Use for multi-hour pipelines; use raw tasks for sub-minute functions.

---
## What's next
**Day 9** → Debugging and Observability — use the Ray Dashboard, `ray.timeline()`, structured logging, and `ray.util.state` to diagnose performance bottlenecks.

Mark Day 8 complete in your [tracker](../index.html).
